# Evaluation-Only Notebook: MIMII-Style Clip AUC Comparison (v0.2)

This notebook is **evaluation-only**:
- loads an existing trained checkpoint
- loads manifests/NPZ windows
- runs inference only
- aggregates clip scores as mean abnormal probability
- computes clip-level ROC AUC and comparison-friendly artifacts

No training, no optimizer, and no LR scheduler are used.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from typing import Sequence
import json
import re

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

try:
    import ipywidgets as widgets
    from ipyfilechooser import FileChooser
except Exception as exc:
    raise ImportError(
        "This notebook requires ipywidgets and ipyfilechooser for checkpoint selection."
    ) from exc

try:
    from IPython.display import display, Markdown
except Exception:
    def display(x):
        print(x)
    class Markdown(str):
        pass

try:
    from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'train').exists() and (candidate / 'preprocessing').exists():
            return candidate
    raise RuntimeError('Could not locate repo root containing train/ and preprocessing/.')


REPO_ROOT = find_repo_root(Path.cwd())
print(f'Using repo root: {REPO_ROOT}')

## 1. Configuration

Select exactly one checkpoint in the FileChooser and click **Select**,
then run this section. v0.2 requires an explicit widget selection.

In [ ]:
def _latest_checkpoint(models_root: Path) -> Path | None:
    candidates = sorted(models_root.glob('*/best_val_model_state_dict.pt'))
    if not candidates:
        candidates = sorted(models_root.glob('*/final_model_state_dict.pt'))
    return candidates[-1].resolve() if candidates else None


def _abs_path(path_value) -> Path:
    p = Path(path_value).expanduser()
    if p.is_absolute():
        return p.resolve()
    return (REPO_ROOT / p).resolve()


def _path_list(values) -> list[Path]:
    if values is None:
        return []
    if isinstance(values, (str, Path)):
        values = [values]
    out = []
    for v in values:
        out.append(_abs_path(v))
    return out


def _load_run_manifest_defaults(checkpoint_path: Path) -> dict:
    run_manifest_path = checkpoint_path.parent / 'run_manifest.json'
    if not run_manifest_path.exists():
        return {
            'window_manifest_paths': [],
            'clip_manifest_paths': [],
            'split_membership_path': None,
        }

    run_manifest = json.loads(run_manifest_path.read_text())
    window_manifest_paths = [_abs_path(p) for p in run_manifest.get('window_manifest_paths', [])]
    clip_manifest_paths = [_abs_path(p) for p in run_manifest.get('clip_manifest_paths', [])]

    split_membership_rel = (
        run_manifest.get('dataset_summary', {}).get('split_artifact_path')
        or run_manifest.get('artifact_paths', {}).get('split_membership')
    )
    split_membership_path = _abs_path(split_membership_rel) if split_membership_rel else None

    return {
        'window_manifest_paths': window_manifest_paths,
        'clip_manifest_paths': clip_manifest_paths,
        'split_membership_path': split_membership_path,
    }


def _get_explicit_checkpoint_selection(chooser: FileChooser) -> Path:
    selected = chooser.selected
    if selected:
        checkpoint_path = Path(selected).expanduser().resolve()
        if checkpoint_path.exists() and checkpoint_path.is_file():
            return checkpoint_path

    # Fallback read from path/filename fields if available and valid.
    selected_path = getattr(chooser, 'selected_path', None)
    selected_filename = getattr(chooser, 'selected_filename', None)
    if selected_path and selected_filename:
        candidate = Path(selected_path) / selected_filename
        candidate = candidate.expanduser().resolve()
        if candidate.exists() and candidate.is_file():
            return candidate

    raise ValueError(
        'No checkpoint has been explicitly selected in the FileChooser. '
        'Choose one .pt file and click "Select", then re-run this cell.'
    )


MODELS_ROOT = REPO_ROOT / 'models'
DEFAULT_CHECKPOINT_PATH = _latest_checkpoint(MODELS_ROOT)
if DEFAULT_CHECKPOINT_PATH is None:
    raise FileNotFoundError('No checkpoint found under models/*/(best_val|final)_model_state_dict.pt')

checkpoint_chooser = FileChooser(str(MODELS_ROOT.resolve()))
checkpoint_chooser.filter_pattern = '*.pt'
checkpoint_chooser.title = '<b>Select EXACTLY one checkpoint (.pt), then click Select</b>'
checkpoint_chooser.reset(path=str(DEFAULT_CHECKPOINT_PATH.parent), filename=DEFAULT_CHECKPOINT_PATH.name)
display(checkpoint_chooser)

# Optional overrides. Keep as None to use run_manifest defaults from the selected checkpoint.
window_manifest_paths_override = None
clip_manifest_paths_override = None
split_membership_path_override = None
npz_search_roots_override = None  # Example: ['training-data']

# Required runtime settings.
split_to_evaluate = 'validation'  # 'validation', 'test', another named split, or 'all'
requested_device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 128
output_root = REPO_ROOT / 'evaluation' / 'outputs'
run_label = ''


def resolve_config() -> dict:
    checkpoint_path = _get_explicit_checkpoint_selection(checkpoint_chooser)

    defaults = _load_run_manifest_defaults(checkpoint_path)

    window_manifest_paths = _path_list(window_manifest_paths_override) if window_manifest_paths_override else defaults['window_manifest_paths']
    clip_manifest_paths = _path_list(clip_manifest_paths_override) if clip_manifest_paths_override else defaults['clip_manifest_paths']

    if not window_manifest_paths:
        raise ValueError('window_manifest_paths is empty. Set overrides or provide run_manifest.json next to checkpoint.')
    if not clip_manifest_paths:
        raise ValueError('clip_manifest_paths is empty. Set overrides or provide run_manifest.json next to checkpoint.')

    for p in [*window_manifest_paths, *clip_manifest_paths]:
        if not Path(p).exists():
            raise FileNotFoundError(f'Manifest not found: {p}')

    if split_membership_path_override is not None:
        split_membership_path = _abs_path(split_membership_path_override)
    else:
        split_membership_path = defaults['split_membership_path']

    if split_membership_path is not None and not split_membership_path.exists():
        raise FileNotFoundError(f'split_membership_path not found: {split_membership_path}')

    npz_search_roots = [
        REPO_ROOT,
        REPO_ROOT / 'training-data',
        REPO_ROOT / 'training-data' / 'tensors',
    ]
    if npz_search_roots_override:
        npz_search_roots.extend(_path_list(npz_search_roots_override))

    runtime_device = torch.device(requested_device)
    if runtime_device.type == 'cuda' and not torch.cuda.is_available():
        print('CUDA requested but unavailable, falling back to CPU.')
        runtime_device = torch.device('cpu')

    cfg = {
        'checkpoint_path': checkpoint_path,
        'window_manifest_paths': [Path(p).resolve() for p in window_manifest_paths],
        'clip_manifest_paths': [Path(p).resolve() for p in clip_manifest_paths],
        'split_membership_path': Path(split_membership_path).resolve() if split_membership_path else None,
        'split_to_evaluate': str(split_to_evaluate).strip(),
        'device': runtime_device,
        'batch_size': int(batch_size),
        'output_root': _abs_path(output_root),
        'run_label': str(run_label).strip(),
        'npz_search_roots': [Path(p).resolve() for p in npz_search_roots],
    }
    return cfg


CONFIG = resolve_config()

print('Resolved configuration:')
print(' - checkpoint_path (selected from widget):', CONFIG['checkpoint_path'])
print(' - split_to_evaluate:', CONFIG['split_to_evaluate'])
print(' - device:', CONFIG['device'])
print(' - batch_size:', CONFIG['batch_size'])
print(' - output_root:', CONFIG['output_root'])
print(' - run_label:', CONFIG['run_label'] or '(none)')
print(' - split_membership_path:', CONFIG['split_membership_path'])
print(' - window_manifest_paths:')
for p in CONFIG['window_manifest_paths']:
    print('   -', p)
print(' - clip_manifest_paths:')
for p in CONFIG['clip_manifest_paths']:
    print('   -', p)

## 2. Model loading

In [ ]:
class Baseline2DCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(2, 16, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(64, 2)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


def load_model(checkpoint_path: Path, device: torch.device):
    checkpoint_obj = torch.load(checkpoint_path, map_location='cpu')

    if isinstance(checkpoint_obj, dict) and 'model_state_dict' in checkpoint_obj:
        state_dict = checkpoint_obj['model_state_dict']
        metadata = {k: v for k, v in checkpoint_obj.items() if k != 'model_state_dict'}
    elif isinstance(checkpoint_obj, dict) and checkpoint_obj and all(torch.is_tensor(v) for v in checkpoint_obj.values()):
        state_dict = checkpoint_obj
        metadata = {'checkpoint_format': 'raw_state_dict'}
    else:
        raise RuntimeError(
            'Unsupported checkpoint format. Expected {"model_state_dict": ...} or raw state_dict dict.'
        )

    model = Baseline2DCNN()
    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()

    return model, checkpoint_obj, metadata


model, checkpoint_payload, checkpoint_metadata = load_model(CONFIG['checkpoint_path'], CONFIG['device'])
input_mode_for_eval = str(checkpoint_metadata.get('input_mode', 'normalized_plus_mask'))

print('Checkpoint loaded:', CONFIG['checkpoint_path'])
print('Model:', model.__class__.__name__)
print('Device:', CONFIG['device'])
print('Eval mode:', not model.training)
print('input_mode_for_eval:', input_mode_for_eval)

print('\nCheckpoint metadata:')
for key in sorted(checkpoint_metadata.keys()):
    if key == 'best_clip_metrics' and isinstance(checkpoint_metadata[key], dict):
        print(f' - {key}: {json.dumps(checkpoint_metadata[key], indent=2)}')
    else:
        print(f' - {key}: {checkpoint_metadata[key]}')

## 3. Dataset loading

In [ ]:
def _read_parquet_many(paths: Sequence[Path], kind_name: str) -> pd.DataFrame:
    frames = []
    for path in paths:
        p = Path(path).resolve()
        if not p.exists():
            raise FileNotFoundError(f'{kind_name} manifest not found: {p}')
        try:
            df = pd.read_parquet(p)
        except Exception as exc:
            raise RuntimeError(
                f'Failed reading {kind_name} parquet at {p}. Ensure pyarrow/fastparquet is installed.'
            ) from exc
        df = df.copy()
        df['_manifest_path'] = str(p)
        frames.append(df)
    if not frames:
        raise RuntimeError(f'No {kind_name} manifests provided.')
    return pd.concat(frames, ignore_index=True)


def _pick_first_column(df: pd.DataFrame, candidates: Sequence[str], required: bool, purpose: str):
    for col in candidates:
        if col in df.columns:
            return col
    if required:
        raise KeyError(f'Missing required column for {purpose}. Tried={list(candidates)}, available={list(df.columns)}')
    return None


def _normalize_path_str(x) -> str:
    return str(x).replace('\\\\', '/').strip()


def _resolve_npz_path(path_value, search_roots: Sequence[Path]) -> Path:
    p = Path(str(path_value)).expanduser()
    candidates = []
    if p.is_absolute():
        candidates.append(p.resolve())
    else:
        for root in search_roots:
            root = Path(root).resolve()
            candidates.append((root / p).resolve())
            candidates.append((root / 'tensors' / p).resolve())
    seen = []
    deduped = []
    for c in candidates:
        key = str(c)
        if key not in seen:
            seen.append(key)
            deduped.append(c)
    for c in deduped:
        if c.exists():
            return c
    return deduped[0] if deduped else p.resolve()


def _label_from_text(text) -> int | None:
    t = str(text).lower()
    if ('abnormal' in t) or ('-ab-' in t) or re.search(r'(^|[-_/])ab($|[-_/])', t):
        return 1
    if ('normal' in t) or ('-nm-' in t) or re.search(r'(^|[-_/])nm($|[-_/])', t):
        return 0
    return None


def _infer_label(row: pd.Series, explicit_label_col: str | None, text_cols: Sequence[str]) -> int | None:
    if explicit_label_col is not None:
        v = row.get(explicit_label_col)
        if pd.notna(v):
            token = str(v).strip().lower()
            if token in {'normal', 'nm', '0', 'false', 'non_anomalous', 'non-anomalous'}:
                return 0
            if token in {'abnormal', 'ab', '1', 'true', 'anomalous', 'anomaly'}:
                return 1
    for c in text_cols:
        if c in row and pd.notna(row[c]):
            parsed = _label_from_text(row[c])
            if parsed is not None:
                return parsed
    return None


def _normalize_split_name(value: str) -> str:
    token = str(value).strip().lower()
    aliases = {
        'val': 'validation',
        'valid': 'validation',
        'validation': 'validation',
        'dev': 'validation',
        'test': 'test',
        'eval': 'test',
        'evaluation': 'test',
        'train': 'train',
        'all': 'all',
    }
    return aliases.get(token, token)


def _validate_window_manifest_alignment(windows_subset_df: pd.DataFrame, subset_name: str):
    required_cols = {'npz_path', 'tensor_index'}
    missing = sorted(required_cols - set(windows_subset_df.columns))
    if missing:
        raise RuntimeError(f'{subset_name} missing required columns for alignment check: {missing}')

    count_mismatches = []
    index_mismatches = []

    for npz_path_value, group in windows_subset_df.groupby('npz_path', sort=False):
        npz_path = Path(str(npz_path_value))
        if not npz_path.exists():
            raise FileNotFoundError(f'NPZ referenced by {subset_name} does not exist: {npz_path}')

        with np.load(npz_path, allow_pickle=False) as npz:
            if 'normalized_window' not in npz.files or 'active_mask' not in npz.files:
                raise KeyError(
                    f'NPZ missing required arrays in {npz_path}. '
                    "Expected keys: 'normalized_window' and 'active_mask'."
                )
            n_normalized = int(npz['normalized_window'].shape[0])
            n_mask = int(npz['active_mask'].shape[0])

        manifest_count = int(len(group))
        if manifest_count != n_normalized or n_normalized != n_mask:
            count_mismatches.append((str(npz_path), manifest_count, n_normalized, n_mask))

        tensor_indices = sorted(group['tensor_index'].astype(int).tolist())
        expected_indices = list(range(len(tensor_indices)))
        if tensor_indices != expected_indices:
            index_mismatches.append((str(npz_path), tensor_indices[:5], tensor_indices[-5:], len(tensor_indices)))

    if count_mismatches:
        preview = '; '.join(
            [f'{p} manifest={m}, normalized_window={n}, active_mask={a}' for p, m, n, a in count_mismatches[:5]]
        )
        raise RuntimeError(
            f'Window/manifest row-count alignment failed for {subset_name}. First mismatches: {preview}'
        )

    if index_mismatches:
        preview = '; '.join(
            [f'{p} first={fst}, last={lst}, count={cnt}' for p, fst, lst, cnt in index_mismatches[:5]]
        )
        raise RuntimeError(
            f'Window tensor_index alignment failed for {subset_name}. Expected contiguous 0..N-1 per npz. '
            f'First mismatches: {preview}'
        )

    return {
        'checked_npz_files': int(windows_subset_df['npz_path'].nunique()),
        'checked_window_rows': int(len(windows_subset_df)),
    }


def load_evaluation_rows(config: dict) -> tuple[pd.DataFrame, dict]:
    window_df = _read_parquet_many(config['window_manifest_paths'], kind_name='window')
    clip_df = _read_parquet_many(config['clip_manifest_paths'], kind_name='clip')

    if 'status' in clip_df.columns:
        clip_df = clip_df[clip_df['status'].isin(['exported', 'skipped_existing'])].copy()

    if window_df.empty or clip_df.empty:
        raise RuntimeError('Window or clip manifest is empty after loading/filtering.')

    clip_npz_col = _pick_first_column(
        clip_df,
        ['tensor_npz_path', 'npz_path', 'tensor_path', 'npz', 'path'],
        required=True,
        purpose='clip npz path',
    )
    window_npz_col = _pick_first_column(
        window_df,
        ['tensor_npz_path', 'npz_path', 'tensor_path', 'npz', 'path'],
        required=True,
        purpose='window npz path',
    )
    tensor_index_col = _pick_first_column(
        window_df,
        ['tensor_index', 'window_index'],
        required=True,
        purpose='window tensor index',
    )

    clip_df = clip_df.copy()
    window_df = window_df.copy()

    clip_df['tensor_npz_path'] = clip_df[clip_npz_col].map(lambda x: str(_resolve_npz_path(x, config['npz_search_roots'])))
    window_df['npz_path'] = window_df[window_npz_col].map(lambda x: str(_resolve_npz_path(x, config['npz_search_roots'])))
    clip_df['_npz_norm'] = clip_df['tensor_npz_path'].map(_normalize_path_str)
    window_df['_npz_norm'] = window_df['npz_path'].map(_normalize_path_str)
    window_df['tensor_index'] = window_df[tensor_index_col].astype(int)

    relative_source_col = _pick_first_column(clip_df, ['relative_source_path'], required=False, purpose='relative source path')
    source_file_col = _pick_first_column(clip_df, ['source_file'], required=False, purpose='source file')
    clip_id_col = _pick_first_column(
        clip_df,
        ['original_clip_id', 'source_clip_id', 'clip_id'],
        required=False,
        purpose='clip id',
    )

    if relative_source_col is not None:
        clip_df['relative_source_path'] = clip_df[relative_source_col].astype(str)
    elif source_file_col is not None:
        clip_df['relative_source_path'] = clip_df[source_file_col].astype(str)
    else:
        clip_df['relative_source_path'] = clip_df['tensor_npz_path'].map(lambda x: Path(str(x)).with_suffix('.wav').as_posix())

    if source_file_col is not None:
        clip_df['source_file'] = clip_df[source_file_col].astype(str)
    else:
        clip_df['source_file'] = clip_df['relative_source_path']

    if clip_id_col is not None:
        clip_df['clip_id'] = clip_df[clip_id_col].astype(str)
    elif relative_source_col is not None:
        clip_df['clip_id'] = clip_df['relative_source_path'].astype(str)
    elif source_file_col is not None:
        clip_df['clip_id'] = clip_df['source_file'].astype(str)
    else:
        clip_df['clip_id'] = clip_df['tensor_npz_path'].map(lambda x: Path(str(x)).with_suffix('.wav').as_posix())

    clip_df['clip_id'] = clip_df['clip_id'].map(_normalize_path_str).map(lambda x: str(x).strip())
    missing_clip_id_mask = clip_df['clip_id'].eq('') | clip_df['clip_id'].str.lower().eq('nan')
    if missing_clip_id_mask.any():
        raise RuntimeError('clip_id resolution failed: at least one clip has empty clip_id.')

    clip_id_to_npz_count = clip_df.groupby('clip_id')['_npz_norm'].nunique()
    clip_id_collision = clip_id_to_npz_count[clip_id_to_npz_count > 1]
    if not clip_id_collision.empty:
        raise RuntimeError(
            'Clip id collision detected: one clip_id maps to multiple tensor_npz_path entries. '
            f'Examples: {clip_id_collision.head(10).to_dict()}'
        )

    npz_to_clip_id_count = clip_df.groupby('_npz_norm')['clip_id'].nunique()
    npz_collision = npz_to_clip_id_count[npz_to_clip_id_count > 1]
    if not npz_collision.empty:
        raise RuntimeError(
            'Manifest consistency error: one tensor_npz_path maps to multiple clip_id values. '
            f'Examples: {npz_collision.head(10).to_dict()}'
        )

    label_col = _pick_first_column(
        clip_df,
        ['label', 'class_label', 'class', 'target', 'condition', 'anomaly_label'],
        required=False,
        purpose='label',
    )
    text_cols_for_inference = [c for c in ['source_file', 'relative_source_path', 'tensor_npz_path'] if c in clip_df.columns]
    clip_df['label'] = clip_df.apply(lambda r: _infer_label(r, label_col, text_cols_for_inference), axis=1)

    missing_label_mask = clip_df['label'].isna()
    if missing_label_mask.any():
        examples = clip_df.loc[missing_label_mask, text_cols_for_inference].head(5)
        raise RuntimeError(
            'Could not infer label for some clips. '
            'Add an explicit label column or ensure paths contain normal/abnormal or -nm-/-ab-. '
            f'Examples:\n{examples.to_string(index=False)}'
        )
    clip_df['label'] = clip_df['label'].astype(int)

    clip_lookup = clip_df[['_npz_norm', 'clip_id', 'label']].drop_duplicates('_npz_norm')

    eval_windows_df = window_df.merge(
        clip_lookup,
        on='_npz_norm',
        how='left',
        validate='many_to_one',
    )

    missing_after_merge = eval_windows_df['clip_id'].isna() | eval_windows_df['label'].isna()
    if missing_after_merge.any():
        bad_rows = eval_windows_df.loc[missing_after_merge, ['npz_path', 'tensor_index']].head(10)
        raise RuntimeError(
            'Window->clip merge produced rows missing clip_id/label. '
            f'Examples:\n{bad_rows.to_string(index=False)}'
        )

    eval_windows_df['clip_id'] = eval_windows_df['clip_id'].astype(str).map(lambda x: x.strip())
    if eval_windows_df['clip_id'].eq('').any() or eval_windows_df['clip_id'].str.lower().eq('nan').any():
        raise RuntimeError('Every sample must have clip_id, but empty clip_id was found in evaluation rows.')

    split_name = _normalize_split_name(config['split_to_evaluate'])
    split_path = config.get('split_membership_path')

    if split_name != 'all':
        if split_path is None:
            raise ValueError(
                'split_to_evaluate is set but split_membership_path is not available. '
                'Set split_membership_path_override or use split_to_evaluate="all".'
            )

        split_df = pd.read_csv(split_path)
        required_split_cols = {'clip_id', 'split'}
        missing_split_cols = sorted(required_split_cols - set(split_df.columns))
        if missing_split_cols:
            raise RuntimeError(f'split_membership is missing required columns: {missing_split_cols}')

        split_df = split_df.copy()
        split_df['clip_id'] = split_df['clip_id'].astype(str).str.strip()
        split_df['_split_norm'] = split_df['split'].map(_normalize_split_name)

        selected_clip_ids = set(split_df.loc[split_df['_split_norm'] == split_name, 'clip_id'].tolist())
        if not selected_clip_ids:
            raise RuntimeError(
                f'No clip_ids matched split_to_evaluate={config["split_to_evaluate"]!r} '
                f'(normalized={split_name!r}) in {split_path}'
            )

        eval_windows_df = eval_windows_df[eval_windows_df['clip_id'].isin(selected_clip_ids)].copy()
        if eval_windows_df.empty:
            raise RuntimeError('Split filtering produced zero evaluation windows.')

    eval_windows_df['label'] = eval_windows_df['label'].astype(int)

    per_clip_label_counts = eval_windows_df.groupby('clip_id')['label'].nunique(dropna=False)
    if (per_clip_label_counts > 1).any():
        bad = per_clip_label_counts[per_clip_label_counts > 1].head(10)
        raise RuntimeError(
            'Grouped windows within a clip must share one true label, but violations were found. '
            f'Examples: {bad.to_dict()}'
        )

    alignment_stats = _validate_window_manifest_alignment(
        eval_windows_df[['npz_path', 'tensor_index']].copy(),
        subset_name=f'eval split={config["split_to_evaluate"]}',
    )

    eval_windows_df = eval_windows_df[['npz_path', 'tensor_index', 'label', 'clip_id']].copy()
    eval_windows_df = eval_windows_df.sort_values(['clip_id', 'npz_path', 'tensor_index']).reset_index(drop=True)

    context = {
        'window_row_count': int(len(eval_windows_df)),
        'clip_count': int(eval_windows_df['clip_id'].nunique()),
        'alignment_stats': alignment_stats,
    }
    return eval_windows_df, context


class TwoChannelWindowDataset(Dataset):
    def __init__(self, rows_df: pd.DataFrame, input_mode_local: str = 'normalized_plus_mask', return_clip_id: bool = True):
        required_cols = {'npz_path', 'tensor_index', 'label', 'clip_id'}
        missing = sorted(required_cols - set(rows_df.columns))
        if missing:
            raise ValueError(f'Dataset rows missing required columns: {missing}')

        if input_mode_local not in {'normalized_plus_mask', 'normalized_only'}:
            raise ValueError(f'Unsupported input_mode: {input_mode_local}')

        self.rows = rows_df[['npz_path', 'tensor_index', 'label', 'clip_id']].reset_index(drop=True).copy()
        self.input_mode = input_mode_local
        self.return_clip_id = bool(return_clip_id)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        npz_path = Path(str(row['npz_path']))
        tensor_index = int(row['tensor_index'])
        label = int(row['label'])
        clip_id = str(row['clip_id']).strip()

        if label not in (0, 1):
            raise ValueError(f'Invalid label at idx={idx}: {label}')
        if not clip_id or clip_id.lower() == 'nan':
            raise ValueError(f'Missing clip_id at idx={idx}')
        if not npz_path.exists():
            raise FileNotFoundError(f'NPZ not found at idx={idx}: {npz_path}')

        with np.load(npz_path, allow_pickle=False) as npz:
            if 'normalized_window' not in npz.files:
                raise KeyError(f"Missing key 'normalized_window' in {npz_path}")
            if 'active_mask' not in npz.files:
                raise KeyError(f"Missing key 'active_mask' in {npz_path}")

            normalized_all = npz['normalized_window']
            active_all = npz['active_mask']

            if tensor_index < 0 or tensor_index >= normalized_all.shape[0]:
                raise IndexError(
                    f'tensor_index out of bounds for {npz_path}: index={tensor_index}, N={normalized_all.shape[0]}'
                )

            normalized_window = normalized_all[tensor_index]
            active_mask = active_all[tensor_index]

        if normalized_window.shape != (96, 64):
            raise ValueError(
                f'normalized_window shape mismatch at idx={idx}: got {normalized_window.shape}, expected (96, 64)'
            )
        if active_mask.shape != (96, 64):
            raise ValueError(
                f'active_mask shape mismatch at idx={idx}: got {active_mask.shape}, expected (96, 64)'
            )

        channel_0 = normalized_window.astype(np.float32, copy=False)
        if self.input_mode == 'normalized_plus_mask':
            channel_1 = active_mask.astype(np.float32, copy=False)
        else:
            channel_1 = np.zeros_like(channel_0, dtype=np.float32)

        x = np.stack([channel_0, channel_1], axis=0)
        if x.shape != (2, 96, 64):
            raise ValueError(f'stacked tensor shape mismatch at idx={idx}: got {x.shape}, expected (2, 96, 64)')

        sample_x = torch.from_numpy(x)
        sample_y = torch.tensor(label, dtype=torch.long)

        if self.return_clip_id:
            return sample_x, sample_y, clip_id
        return sample_x, sample_y


eval_windows_df, dataset_context = load_evaluation_rows(CONFIG)

pin_memory = CONFIG['device'].type == 'cuda'
eval_dataset = TwoChannelWindowDataset(
    eval_windows_df,
    input_mode_local=input_mode_for_eval,
    return_clip_id=True,
)
eval_loader = DataLoader(
    eval_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory,
)

sample_x, sample_y, sample_clip_id = eval_dataset[0]

print('Dataset ready for evaluation:')
print(' - window_row_count:', dataset_context['window_row_count'])
print(' - clip_count:', dataset_context['clip_count'])
print(' - alignment_stats:', dataset_context['alignment_stats'])
print(' - sample tensor shape:', tuple(sample_x.shape))
print(' - sample label:', int(sample_y))
print(' - sample clip_id:', sample_clip_id)

display(eval_windows_df.head(10))

## 4. Inference loop

In [ ]:
def run_inference(model: nn.Module, loader: DataLoader, device: torch.device) -> list[dict]:
    model.eval()
    records = []

    with torch.no_grad():
        for batch in loader:
            if len(batch) != 3:
                raise RuntimeError('Expected evaluation batch as (x, y, clip_id).')

            x, y, clip_ids = batch
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            abnormal_probs = probs[:, 1]
            preds = torch.argmax(logits, dim=1)

            y_cpu = y.detach().cpu().tolist()
            pred_cpu = preds.detach().cpu().tolist()
            abnormal_cpu = abnormal_probs.detach().cpu().tolist()
            clip_ids_list = [str(c).strip() for c in clip_ids]

            if len(clip_ids_list) != len(y_cpu):
                raise RuntimeError(
                    f'clip_id batch size mismatch: clip_ids={len(clip_ids_list)}, labels={len(y_cpu)}'
                )

            for clip_id, true_label, pred_class, abnormal_prob in zip(
                clip_ids_list,
                y_cpu,
                pred_cpu,
                abnormal_cpu,
            ):
                if not clip_id or clip_id.lower() == 'nan':
                    raise RuntimeError('Encountered empty clip_id during inference.')
                records.append(
                    {
                        'clip_id': clip_id,
                        'true_label': int(true_label),
                        'abnormal_prob': float(abnormal_prob),
                        'pred_class': int(pred_class),
                    }
                )

    if not records:
        raise RuntimeError('Inference produced zero window records.')
    return records


window_records = run_inference(model, eval_loader, CONFIG['device'])
window_records_df = pd.DataFrame(window_records)

print('Inference complete.')
print(' - window records:', len(window_records_df))
print(' - unique clips:', window_records_df['clip_id'].nunique())
display(window_records_df.head(10))

## 5. Clip aggregation

In [ ]:
def aggregate_clip_scores(window_records: list[dict]) -> pd.DataFrame:
    records_df = pd.DataFrame(window_records).copy()
    required_cols = {'clip_id', 'true_label', 'abnormal_prob'}
    missing = sorted(required_cols - set(records_df.columns))
    if missing:
        raise RuntimeError(f'Window records missing required columns: {missing}')

    records_df['clip_id'] = records_df['clip_id'].astype(str)
    records_df['true_label'] = records_df['true_label'].astype(int)
    records_df['abnormal_prob'] = records_df['abnormal_prob'].astype(float)

    if records_df['clip_id'].isna().any() or records_df['clip_id'].str.strip().eq('').any():
        raise RuntimeError('At least one window record has an empty clip_id.')

    per_clip_rows = []
    for clip_id, group in records_df.groupby('clip_id', sort=True, dropna=False):
        unique_labels = sorted(group['true_label'].unique().tolist())
        if len(unique_labels) != 1:
            raise RuntimeError(f'Clip {clip_id} has inconsistent labels across windows: {unique_labels}')

        true_label = int(unique_labels[0])
        num_windows = int(len(group))
        mean_abnormal_prob = float(group['abnormal_prob'].mean())
        clip_pred_label = int(1 if mean_abnormal_prob >= 0.5 else 0)
        clip_correct = int(clip_pred_label == true_label)

        per_clip_rows.append(
            {
                'clip_id': str(clip_id),
                'true_label': true_label,
                'num_windows': num_windows,
                'mean_abnormal_prob': mean_abnormal_prob,
                'clip_pred_label': clip_pred_label,
                'clip_correct': clip_correct,
            }
        )

    per_clip_df = pd.DataFrame(per_clip_rows).sort_values('clip_id').reset_index(drop=True)
    if per_clip_df.empty:
        raise RuntimeError('Clip aggregation produced zero rows.')
    return per_clip_df


per_clip_scores_df = aggregate_clip_scores(window_records)
print('Clip aggregation complete.')
print(' - clip_count:', len(per_clip_scores_df))
display(per_clip_scores_df.head(10))

## 6. Metrics

In [ ]:
def _binary_confusion_matrix_numpy(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    cm = np.zeros((2, 2), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        if t in (0, 1) and p in (0, 1):
            cm[int(t), int(p)] += 1
    return cm


def compute_metrics(per_clip_df: pd.DataFrame, window_records: list[dict], include_window_auc: bool = True):
    y_clip_true = per_clip_df['true_label'].to_numpy(dtype=np.int64)
    y_clip_pred = per_clip_df['clip_pred_label'].to_numpy(dtype=np.int64)
    y_clip_score = per_clip_df['mean_abnormal_prob'].to_numpy(dtype=np.float64)

    clip_count = int(len(per_clip_df))
    clip_accuracy = float(per_clip_df['clip_correct'].mean()) if clip_count else 0.0

    clip_auc = None
    if SKLEARN_AVAILABLE and len(np.unique(y_clip_true)) == 2:
        clip_auc = float(roc_auc_score(y_clip_true, y_clip_score))

    if SKLEARN_AVAILABLE:
        clip_cm = confusion_matrix(y_clip_true, y_clip_pred, labels=[0, 1])
    else:
        clip_cm = _binary_confusion_matrix_numpy(y_clip_true, y_clip_pred)

    clip_report_dict = None
    clip_report_text = None
    if SKLEARN_AVAILABLE:
        clip_report_dict = classification_report(
            y_clip_true,
            y_clip_pred,
            labels=[0, 1],
            target_names=['normal', 'abnormal'],
            output_dict=True,
            zero_division=0,
        )
        clip_report_text = classification_report(
            y_clip_true,
            y_clip_pred,
            labels=[0, 1],
            target_names=['normal', 'abnormal'],
            digits=4,
            zero_division=0,
        )

    pred_normal_count = int((y_clip_pred == 0).sum())
    pred_abnormal_count = int((y_clip_pred == 1).sum())
    pred_normal_prop = float(pred_normal_count / clip_count) if clip_count else 0.0
    pred_abnormal_prop = float(pred_abnormal_count / clip_count) if clip_count else 0.0

    window_auc = None
    if include_window_auc and SKLEARN_AVAILABLE:
        window_df = pd.DataFrame(window_records)
        if not window_df.empty and len(window_df['true_label'].unique()) == 2:
            window_auc = float(
                roc_auc_score(
                    window_df['true_label'].to_numpy(dtype=np.int64),
                    window_df['abnormal_prob'].to_numpy(dtype=np.float64),
                )
            )

    metrics = {
        'checkpoint_path': str(CONFIG['checkpoint_path']),
        'split_to_evaluate': CONFIG['split_to_evaluate'],
        'clip_count': clip_count,
        'clip_accuracy': clip_accuracy,
        'clip_auc': clip_auc,
        'predicted_normal_clip_count': pred_normal_count,
        'predicted_abnormal_clip_count': pred_abnormal_count,
        'predicted_normal_clip_proportion': pred_normal_prop,
        'predicted_abnormal_clip_proportion': pred_abnormal_prop,
        'window_auc_optional': window_auc,
    }

    cm_df = pd.DataFrame(
        clip_cm,
        index=['true_normal', 'true_abnormal'],
        columns=['pred_normal', 'pred_abnormal'],
    )

    return metrics, clip_cm, cm_df, clip_report_text, clip_report_dict


metrics_payload, clip_cm_array, clip_cm_df, clip_report_text, clip_report_dict = compute_metrics(
    per_clip_scores_df,
    window_records,
    include_window_auc=True,
)

print('clip_count:', metrics_payload['clip_count'])
print('clip_accuracy:', f"{metrics_payload['clip_accuracy']:.4f}")
print('clip_auc:', metrics_payload['clip_auc'])
print('predicted normal clips:', metrics_payload['predicted_normal_clip_count'])
print('predicted abnormal clips:', metrics_payload['predicted_abnormal_clip_count'])
print('predicted normal proportion:', f"{metrics_payload['predicted_normal_clip_proportion']:.4f}")
print('predicted abnormal proportion:', f"{metrics_payload['predicted_abnormal_clip_proportion']:.4f}")
print('window_auc (optional):', metrics_payload['window_auc_optional'])
print('clip confusion matrix:')
display(clip_cm_df)

if clip_report_text is not None:
    print('clip classification report:')
    print(clip_report_text)
else:
    print('sklearn.metrics unavailable; classification report skipped.')

## 7. Output artifacts

In [ ]:
def _slugify(text: str) -> str:
    cleaned = re.sub(r'[^a-zA-Z0-9._-]+', '-', str(text).strip())
    cleaned = cleaned.strip('-_.')
    return cleaned.lower()


def save_artifacts(
    per_clip_df: pd.DataFrame,
    metrics: dict,
    output_root_path: Path,
    checkpoint_path: Path,
    run_label_local: str = '',
    confusion_matrix_array: np.ndarray | None = None,
    classification_report_dict_local: dict | None = None,
) -> dict:
    output_root_path = Path(output_root_path).resolve()
    output_root_path.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
    suffix = f"-{_slugify(run_label_local)}" if str(run_label_local).strip() else ''
    run_dir = output_root_path / f'{timestamp}-mimii-eval{suffix}'
    run_dir.mkdir(parents=True, exist_ok=False)

    per_clip_path = run_dir / 'per_clip_scores.csv'
    metrics_path = run_dir / 'metrics.json'
    confusion_matrix_path = run_dir / 'confusion_matrix.csv'
    classification_report_path = run_dir / 'classification_report.json'

    per_clip_df.to_csv(per_clip_path, index=False)

    metrics_to_save = dict(metrics)
    metrics_to_save['checkpoint_path'] = str(Path(checkpoint_path).resolve())
    metrics_to_save['saved_at_utc'] = datetime.now(timezone.utc).isoformat()
    metrics_path.write_text(json.dumps(metrics_to_save, indent=2))

    if confusion_matrix_array is not None:
        cm_df = pd.DataFrame(
            confusion_matrix_array,
            index=['true_normal', 'true_abnormal'],
            columns=['pred_normal', 'pred_abnormal'],
        )
        cm_df.to_csv(confusion_matrix_path, index=True)
    else:
        confusion_matrix_path = None

    if classification_report_dict_local is not None:
        classification_report_path.write_text(json.dumps(classification_report_dict_local, indent=2))
    else:
        classification_report_path = None

    return {
        'run_dir': run_dir,
        'per_clip_scores_csv': per_clip_path,
        'metrics_json': metrics_path,
        'confusion_matrix_csv': confusion_matrix_path,
        'classification_report_json': classification_report_path,
    }


artifact_paths = save_artifacts(
    per_clip_df=per_clip_scores_df,
    metrics=metrics_payload,
    output_root_path=CONFIG['output_root'],
    checkpoint_path=CONFIG['checkpoint_path'],
    run_label_local=CONFIG['run_label'],
    confusion_matrix_array=clip_cm_array,
    classification_report_dict_local=clip_report_dict,
)

print('Artifacts saved:')
for key, value in artifact_paths.items():
    print(f' - {key}: {value}')

## 8. MIMII-baseline comparison readiness

- Our score uses **mean abnormal probability per clip** (`mean_abnormal_prob`).
- Our comparison metric is **clip-level ROC AUC** (`clip_auc`).
- This setup is intended to align with the published **MIMII baseline evaluation style**.
- This does **not** imply our training method matches the baseline training method.